# Street Fighter II — Student Guide · Group 10 Counter-Ryu
**SYE6308 | 2026-11-10 | 基于课程原版的训练、评估与对打 notebook**

本文件直接从 `notebooks/upstream/sye6308-sf2-classroom-2026.ipynb` 的 **53 个单元格**修改而来，保留原章节及 PvE/PvP 流程。我们的观察、动作、奖励、循环策略分别接入 **3.1–3.4** 和 **9.2a–9.2d**；训练使用现有循环 PPO 模块，不另写一套训练器。

| 目的 | 原课程章节 | 本版行为 |
|---|---|---|
| 检查、观看当前模型 | 1 → 2 → 3 → 4 → 6 → 8 | 默认“全部运行”会现场完成 2 场 PvE、2 场 PvP并显示录像 |
| 继续训练当前模型 | 1–4 → 5 → 6 | 打开第5节训练开关；更新新的工作副本 |
| 与保存的对手训练 | 1 → 9.1 → 9.2全部四格 → 9.3 → 9.4 → 9.5 | 同一份当前权重继续学习，可同时采样 CPU 保留任务 |
| 导出模型 | 7 → 7.2 → 7.3 | 本地推理导出、独立加载检查及文件链接 |

当前默认权重仍为 **F / feedback_candidate**。它已有96/96 CPU KO、204/204已知PvP和90/90保护记录；五天研究中的 C 在共同后续80条件为59胜，F为61胜，未显示整体提升。局部走位和连续应对已经出现，但稳定的可积累改进尚未实现。

F 是**循环 PPO基础 + 经完整对局验证的连续轨迹监督整合**。第5节提供真实循环PPO续训，第5.1节另保留历史轨迹整合复现入口，二者不混称。默认训练开关与原课程一样关闭；只把原始模型加载并展示，不会偷偷更新权重。


## 1. Shared setup — required for every route
**按顺序运行接下来三个代码单元格。** 保留课程的目录设置、公共运行模块和依赖检查三步。Mac、Windows、Kaggle上均从完整仓库解析路径，不再写入只适用于Kaggle的 `/kaggle/working`。

原公共核心同时包含ROM扫描、文件摘要和课程打包检查。遵守本项目禁止文件哈希的要求，本版使用已验证的共享模拟器、评估器和本地导出入口；游戏规则和裁判不改。`notebooks/upstream`保持只读。

学生文件仍写入独立的 `student/pve/` 与 `student/pvp/` 工作目录。当前内核须为已安装依赖的`sye6308`；不自动重装模拟器或系统组件。每次运行创建新的输出目录。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import copy, importlib.util, json, os, sys

starts = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPO_ROOT = next((p for p in starts + [p/'github_sync' for p in starts]
                  if (p/'experiments/league_adaptive/runtime.py').is_file()
                  and (p/'models/ryu_targeted_delivery/feedback_runtime/policy.zip').is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError('请在完整github_sync仓库中打开本notebook。')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
ROOT = REPO_ROOT/'artifacts/group10_classroom_1110'/stamp
CODE_DIR = REPO_ROOT/'experiments/league_adaptive'
PVE_STUDENT_DIR = ROOT/'student/pve'
PVP_STUDENT_DIR = ROOT/'student/pvp'
for directory in (ROOT, PVE_STUDENT_DIR, PVP_STUDENT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS'):
    os.environ[name] = '1'
os.environ['PYTHONUTF8'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ.setdefault('MPLCONFIGDIR', str(ROOT/'matplotlib'))
DEFAULT_AGENT = REPO_ROOT/'models/ryu_targeted_delivery/feedback_runtime/policy.zip'
CURRENT_AGENT = DEFAULT_AGENT
POST_MATCH_SECONDS = 1.0
COMMON_RULES = dict(ruleset='hyper', speed_stars=2, decision_frames=6,
                    position_scale=1024.0, max_round_frames=7200)
print('仓库：', REPO_ROOT)
print('本次工作目录：', ROOT)
print('当前模型：', CURRENT_AGENT)


In [ ]:
# Public core: 保留课程的公共模块位置，复用现有实现。
# 不执行原版内嵌安装/摘要/提交代码，不复制trainer或裁判。
from experiments.league_adaptive.classroom import (
    new_run, prepare_student, check_environment, train_ppo, evaluate_cases,
    show_result, show_training_review, export_model, reproduce_integration,
)
from experiments.league_adaptive.runtime import COMMON, load_actor, setup
from experiments.league_adaptive.characters import CHARACTERS
from IPython.display import display, Markdown, FileLink

if COMMON_RULES != COMMON:
    raise RuntimeError('Notebook与共享模拟器规则不一致，请核对课程规则。')
LAST_TRAINING = LAST_PVP_TRAINING = None
TRAIN_REPORT = PVP_TRAIN_REPORT = None
LAST_SUBMISSION = None  # 本版指本地推理目录，不伪称官方提交包
print('Shared setup ready：使用同一模拟器、循环策略、训练器与评估器。')


In [ ]:
required = ['numpy','torch','stable_baselines3','cv2','stable_retro','imageio_ffmpeg']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f'当前内核缺少{missing}；请切换已经配置的sye6308环境。')
import torch, stable_baselines3, stable_retro
setup()
print('Setup finished')
print({'python': sys.executable, 'torch': torch.__version__,
       'stable_baselines3': stable_baselines3.__version__,
       'stable_retro': getattr(stable_retro,'__version__','installed'),
       'training_device': 'cpu', 'current_model': str(DEFAULT_AGENT.relative_to(REPO_ROOT))})
print('本版调用已验证的CPU循环训练实现；GPU不会因可用而自动启用。')


## 2. PvE settings — agent versus the built-in CPU
保留课程参数分组。己方固定为当前权重对应的Ryu；对手可覆盖12角色×8星。`speed_stars=2`是游戏速度，`difficulty_stars`才是CPU难度。

训练配置中的 `n_envs`、`n_steps`、`batch_size`、`n_epochs`、学习率、熵和裁剪参数会实际传入循环训练器。`batch_size`必须是`sequence_length`的整数倍；一次完整rollout包含`n_envs*n_steps`次决策。每次决策通常执行6原生帧，临近终局可能更少。

当前共享循环实现使用`gamma=1`和CPU，不能改一个字段就声称实现CUDA/MPS或其他折扣机制。CPU开局默认offset0；PvP另在第9节配置。默认第6节只演示八星Ryu/Sagat，完整96条件评估可显式开启。


In [ ]:
PVE_AGENT_CONFIG = dict(team_name='group10', character='Ryu', seed=111031, use_position=True)
PVE_CONFIG = dict(opponent_characters=list(CHARACTERS), difficulty_stars=list(range(1,9)),
                  neutral_jitter_max=0)
PVE_TRAIN_CONFIG = dict(
    total_timesteps=2048, n_envs=4, n_steps=512, sequence_length=32,
    batch_size=256, n_epochs=4, learning_rate=1e-5,
    gamma=1.0, gae_lambda=0.95, clip_range=0.1, ent_coef=0.005,
    target_kl=0.04, device='cpu', checkpoint_every=2048, max_seconds=1800,
    seed=PVE_AGENT_CONFIG['seed'], coverage_offsets=[0,12],
    training_group_weights=dict(pve=1.0,p1=0.0,p2=0.0),
)
PVE_VALIDATION_OFFSETS = [0]
PVE_POOL = [dict(id=f'CPU_{c}_{s}',kind='cpu',character=c,stars=s)
            for c in PVE_CONFIG['opponent_characters'] for s in PVE_CONFIG['difficulty_stars']]
if PVE_AGENT_CONFIG['character'] != 'Ryu' or not PVE_AGENT_CONFIG['use_position']:
    raise ValueError('当前F模型为Ryu且需要坐标输入；不能仅改角色/观察开关后复用权重。')
if PVE_CONFIG['neutral_jitter_max'] != 0:
    raise ValueError('当前CPU入口固定offset0；不能将未接入的jitter当作有效设置。')
print('Agent:', PVE_AGENT_CONFIG)
print('CPU conditions:', len(PVE_POOL))
print('Training:', PVE_TRAIN_CONFIG)


## 3. PvE student files — run all four code cells
仍按课程设计执行四个`%%writefile`单元格，分别保存观察、动作、奖励和模型配置到`student/pve/`。第4节会读取这四个文件，构造实际用于检查、训练及对打的工作模型；修改后必须重跑第4节。

### 3.1 Observation — what the agent sees
Group10将原版168×192输入改为84×96，保留四次决策画面、裁剪检查和六维自身优先坐标。不是四个相邻原生帧；六帧决策间隔不变。修改输入尺寸后不能强行加载不兼容F权重。


In [ ]:
%%writefile $PVE_STUDENT_DIR/student_observation.py
"""Student-editable image input. Default: four 96 x 84 grayscale frames.

Only transform the supplied RGB image. Never operate the game or change rewards.
The coordinate branch still uses the six original self-first position values.
SB3 divides these uint8 images by 255 once, inside its policy preprocessor.
"""
import cv2
import numpy as np

INPUT_CONFIG = {'color': 'gray', 'width': 96, 'height': 84, 'stack_frames': 4, 'raw_shape': [224, 320, 3], 'crop_xyxy': [0, 0, 256, 224], 'verify_right_padding': True, 'black_threshold': 5, 'max_nonblack_fraction': 0.005}


def preprocess(rgb):
    """One RGB uint8 frame -> (channels, height, width) uint8 array."""
    a = np.asarray(rgb)
    c = INPUT_CONFIG
    if a.dtype != np.uint8 or tuple(a.shape) != tuple(c["raw_shape"]):
        raise ValueError(f"Unexpected raw image: {a.shape} {a.dtype}; expected {c['raw_shape']} uint8.")
    x0, y0, x1, y1 = c["crop_xyxy"]
    if c["verify_right_padding"] and x1 < a.shape[1]:
        excluded = a[:, x1:, :]
        fraction = float(np.mean(np.max(excluded, axis=2) > c["black_threshold"]))
        if fraction > c["max_nonblack_fraction"]:
            raise ValueError(
                f"CROP_GUARD_FAILED: right-side nonblack fraction={fraction:.6f}. "
                "The fixed viewport is not valid for this observed frame. "
                "Stop and inspect the raw image; do not train with a silently changing crop."
            )
    view = a[y0:y1, x0:x1, :]
    if c["color"] == "gray":
        gray = cv2.cvtColor(view, cv2.COLOR_RGB2GRAY)
        result = cv2.resize(gray, (c["width"], c["height"]), interpolation=cv2.INTER_AREA)[None, :, :]
    else:
        result = cv2.resize(view, (c["width"], c["height"]), interpolation=cv2.INTER_AREA).transpose(2, 0, 1)
    return np.ascontiguousarray(result, dtype=np.uint8)


### 3.2 Actions — what one action ID means
保留课程15个基础输入，加入当前模型使用的49个短模板，共64个部署动作。每个模板恰好6帧；连续相同按键表示持续按住，不是自动松开再按。

以下定义来自F已验证的部署目录，保持编号含义。修改或新增输入后应显式从头初始化，不能重新解释旧权重输出。名字只表示请求输入；`DP`/`QCF`不保证实际触发对应技能，方向输入也不等于有效走位。


In [ ]:
%%writefile $PVE_STUDENT_DIR/student_actions.py
"""Group10: 15 course basics plus the 49 deployed six-frame inputs."""
CUSTOM_ACTIONS = [{'name': 'NO_OP_X_tap',
  'segments': [{'keys': [], 'frames': 2}, {'keys': ['X'], 'frames': 2}, {'keys': [], 'frames': 2}]},
 {'name': 'NO_OP_Y_tap',
  'segments': [{'keys': [], 'frames': 2}, {'keys': ['Y'], 'frames': 2}, {'keys': [], 'frames': 2}]},
 {'name': 'NO_OP_Z_tap',
  'segments': [{'keys': [], 'frames': 2}, {'keys': ['Z'], 'frames': 2}, {'keys': [], 'frames': 2}]},
 {'name': 'NO_OP_A_tap',
  'segments': [{'keys': [], 'frames': 2}, {'keys': ['A'], 'frames': 2}, {'keys': [], 'frames': 2}]},
 {'name': 'NO_OP_B_tap',
  'segments': [{'keys': [], 'frames': 2}, {'keys': ['B'], 'frames': 2}, {'keys': [], 'frames': 2}]},
 {'name': 'NO_OP_C_tap',
  'segments': [{'keys': [], 'frames': 2}, {'keys': ['C'], 'frames': 2}, {'keys': [], 'frames': 2}]},
 {'name': 'LEFT_X_tap',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'X'], 'frames': 2},
               {'keys': ['LEFT'], 'frames': 2}]},
 {'name': 'LEFT_Y_tap',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'Y'], 'frames': 2},
               {'keys': ['LEFT'], 'frames': 2}]},
 {'name': 'LEFT_Z_tap',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'Z'], 'frames': 2},
               {'keys': ['LEFT'], 'frames': 2}]},
 {'name': 'LEFT_A_tap',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'A'], 'frames': 2},
               {'keys': ['LEFT'], 'frames': 2}]},
 {'name': 'LEFT_B_tap',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'B'], 'frames': 2},
               {'keys': ['LEFT'], 'frames': 2}]},
 {'name': 'LEFT_C_tap',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'C'], 'frames': 2},
               {'keys': ['LEFT'], 'frames': 2}]},
 {'name': 'RIGHT_X_tap',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'X'], 'frames': 2},
               {'keys': ['RIGHT'], 'frames': 2}]},
 {'name': 'RIGHT_Y_tap',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'Y'], 'frames': 2},
               {'keys': ['RIGHT'], 'frames': 2}]},
 {'name': 'RIGHT_Z_tap',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'Z'], 'frames': 2},
               {'keys': ['RIGHT'], 'frames': 2}]},
 {'name': 'RIGHT_A_tap',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'A'], 'frames': 2},
               {'keys': ['RIGHT'], 'frames': 2}]},
 {'name': 'RIGHT_B_tap',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'B'], 'frames': 2},
               {'keys': ['RIGHT'], 'frames': 2}]},
 {'name': 'RIGHT_C_tap',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'C'], 'frames': 2},
               {'keys': ['RIGHT'], 'frames': 2}]},
 {'name': 'DOWN_X_tap',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'X'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2}]},
 {'name': 'DOWN_Y_tap',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'Y'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2}]},
 {'name': 'DOWN_Z_tap',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'Z'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2}]},
 {'name': 'DOWN_A_tap',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'A'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2}]},
 {'name': 'DOWN_B_tap',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'B'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2}]},
 {'name': 'DOWN_C_tap',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'C'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2}]},
 {'name': 'UP_X_tap',
  'segments': [{'keys': ['UP'], 'frames': 2},
               {'keys': ['UP', 'X'], 'frames': 2},
               {'keys': ['UP'], 'frames': 2}]},
 {'name': 'UP_Y_tap',
  'segments': [{'keys': ['UP'], 'frames': 2},
               {'keys': ['UP', 'Y'], 'frames': 2},
               {'keys': ['UP'], 'frames': 2}]},
 {'name': 'UP_Z_tap',
  'segments': [{'keys': ['UP'], 'frames': 2},
               {'keys': ['UP', 'Z'], 'frames': 2},
               {'keys': ['UP'], 'frames': 2}]},
 {'name': 'UP_A_tap',
  'segments': [{'keys': ['UP'], 'frames': 2},
               {'keys': ['UP', 'A'], 'frames': 2},
               {'keys': ['UP'], 'frames': 2}]},
 {'name': 'UP_B_tap',
  'segments': [{'keys': ['UP'], 'frames': 2},
               {'keys': ['UP', 'B'], 'frames': 2},
               {'keys': ['UP'], 'frames': 2}]},
 {'name': 'UP_C_tap',
  'segments': [{'keys': ['UP'], 'frames': 2},
               {'keys': ['UP', 'C'], 'frames': 2},
               {'keys': ['UP'], 'frames': 2}]},
 {'name': 'UP_LEFT_X_tap',
  'segments': [{'keys': ['UP', 'LEFT'], 'frames': 2},
               {'keys': ['UP', 'LEFT', 'X'], 'frames': 2},
               {'keys': ['UP', 'LEFT'], 'frames': 2}]},
 {'name': 'UP_LEFT_Y_tap',
  'segments': [{'keys': ['UP', 'LEFT'], 'frames': 2},
               {'keys': ['UP', 'LEFT', 'Y'], 'frames': 2},
               {'keys': ['UP', 'LEFT'], 'frames': 2}]},
 {'name': 'UP_LEFT_Z_tap',
  'segments': [{'keys': ['UP', 'LEFT'], 'frames': 2},
               {'keys': ['UP', 'LEFT', 'Z'], 'frames': 2},
               {'keys': ['UP', 'LEFT'], 'frames': 2}]},
 {'name': 'UP_RIGHT_X_tap',
  'segments': [{'keys': ['UP', 'RIGHT'], 'frames': 2},
               {'keys': ['UP', 'RIGHT', 'X'], 'frames': 2},
               {'keys': ['UP', 'RIGHT'], 'frames': 2}]},
 {'name': 'UP_RIGHT_Y_tap',
  'segments': [{'keys': ['UP', 'RIGHT'], 'frames': 2},
               {'keys': ['UP', 'RIGHT', 'Y'], 'frames': 2},
               {'keys': ['UP', 'RIGHT'], 'frames': 2}]},
 {'name': 'UP_RIGHT_Z_tap',
  'segments': [{'keys': ['UP', 'RIGHT'], 'frames': 2},
               {'keys': ['UP', 'RIGHT', 'Z'], 'frames': 2},
               {'keys': ['UP', 'RIGHT'], 'frames': 2}]},
 {'name': 'NO_OP_PPP',
  'segments': [{'keys': ['X', 'Y', 'Z'], 'frames': 2}, {'keys': [], 'frames': 4}]},
 {'name': 'QCF_RIGHT_Y',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'Y'], 'frames': 2}]},
 {'name': 'QCF_RIGHT_B',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'B'], 'frames': 2}]},
 {'name': 'QCF_RIGHT_C',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'C'], 'frames': 2}]},
 {'name': 'DP_RIGHT_X',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'RIGHT', 'X'], 'frames': 2}]},
 {'name': 'DP_RIGHT_Z',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'RIGHT', 'Z'], 'frames': 2}]},
 {'name': 'DP_RIGHT_B',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'RIGHT', 'B'], 'frames': 2}]},
 {'name': 'QCF_LEFT_Y',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'Y'], 'frames': 2}]},
 {'name': 'QCF_LEFT_B',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'B'], 'frames': 2}]},
 {'name': 'QCF_LEFT_C',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'C'], 'frames': 2}]},
 {'name': 'DP_LEFT_X',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'LEFT', 'X'], 'frames': 2}]},
 {'name': 'DP_LEFT_Z',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'LEFT', 'Z'], 'frames': 2}]},
 {'name': 'DP_LEFT_B',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'LEFT', 'B'], 'frames': 2}]}]


### 3.3 Reward — feedback for training
保留原课程`compute_reward(t)`接口及默认奖励：净伤害的0.5倍归一化反馈，加自然胜负奖励。这里修改的代码会实际用于第5节训练，P2训练时伤害与胜负也以学习者为准；不会改变原生裁判。

F过去各阶段使用过不同的研究奖励和监督整合；加载现有F不受本奖励影响。本节定义的是此后新PPO训练阶段的奖励，不能把它说成完整复现F历史训练。


In [ ]:
%%writefile $PVE_STUDENT_DIR/student_reward.py
"""Simple classroom reward: small net-damage feedback + clear match outcome.

Compatible with SF2_Flexible_Classroom_PPO (ordinary PPO, six-frame steps).
The arena supplies damage_dealt/damage_taken with negative KO HP treated as 0.
A natural timeout uses the arena's win/loss/draw result, just like a KO.
A safety cutoff is not a loss. No distance, survival, action or time bonus.

With <=176 HP per fighter and no in-round healing, the sum of damage rewards
is bounded by [-0.5, 0.5]. A normal completed win therefore has positive raw
episode return, and a loss negative raw episode return. This is NOT a claim
about discounted returns, win rate, or empirically verified learning success.
"""
import math

REWARD_VERSION = "net_damage_outcome_v1"
FULL_HP = 176.0
DAMAGE_WEIGHT = 0.5
OUTCOME_BONUS = {
    "ongoing": 0.0,
    "win": 1.0,
    "loss": -1.0,
    "draw": 0.0,
    "cutoff": 0.0,
}


def compute_reward(t: dict) -> float:
    dealt = float(t["damage_dealt"])
    taken = float(t["damage_taken"])
    result = t["result"]

    # Do not silently replace invalid readings with a plausible reward.
    if not (math.isfinite(dealt) and math.isfinite(taken)):
        raise ValueError("Damage must be finite.")
    if not (0.0 <= dealt <= FULL_HP and 0.0 <= taken <= FULL_HP):
        raise ValueError("Damage is outside the expected 0..176 range.")
    if result not in OUTCOME_BONUS:
        raise ValueError(f"Unknown match result: {result!r}")

    # Count this step's damage even on the final step, then add outcome once.
    damage_reward = DAMAGE_WEIGHT * (dealt - taken) / FULL_HP
    return float(damage_reward + OUTCOME_BONUS[result])


### 3.4 Model — how observations become decisions
原课程此处是普通CNN-PPO；Group10接入已有`SequencePPOPolicy`：视觉与坐标表示，加实际输入历史和128维GRU，再由非线性动作输出层决策。四帧图像在特征提取中还形成相邻差分。部署接口为64个输入，F内部保留125维输出与动作历史映射。

本cell保存实际构造模型时读取的配置。CNN/GRU前向实现复用`experiments/ryu_counterq/policy.py`中的`SequencePPOPolicy`及共享编码器；文件夹名字含Q，不表示这里在运行DQN。训练用PPO策略分布和GAE，不以Q目标替代PPO。

架构或历史语义变化需要兼容检查/从头初始化；不能只改维度后继续使用F。当前加载F；尚未实现上一轮讨论的DCARL式持续改进。


In [ ]:
%%writefile $PVE_STUDENT_DIR/student_model.py
"""Group10 recurrent PPO model settings used by prepare_student and the trainer."""
import copy

MODEL_CONFIG = {
    'kind': 'sequence_ppo',
    'memory_hidden': 128,
    'trainable_visual': True,
    'physical_actions': True,
    'signed_relative_position': True,
    'facing_canonical': True,
    'action_readout': 'relu',
    'action_temperature': 1.0,
    'input_history': {'mode': 'real', 'cap_native_frames': 120, 'normalization': 120},
}

def configure_definition(definition):
    result = copy.deepcopy(definition)
    result.update(copy.deepcopy(MODEL_CONFIG))
    return result


## 4. PvE environment check — required before training
保留原课程检查位置：读取四个学生文件，生成工作副本，显示原始画面、四帧输入和步进后的历史，检查动作及奖励。检查首个CPU条件，不把检查通过当作96条件战斗通过。

`PVE_START_FROM`为空时使用当前模型。默认保留F权重；`PVE_START_SCRATCH=True`才会随机初始化。工作副本不会覆盖模型目录。


In [ ]:
RUN_ENVIRONMENT_CHECK = True
PVE_START_FROM = ''
PVE_START_SCRATCH = False
if RUN_ENVIRONMENT_CHECK:
    phase = new_run(ROOT,'pve_check')
    PVE_INITIAL = prepare_student(PVE_STUDENT_DIR, PVE_START_FROM or CURRENT_AGENT,
                                  phase/'student', scratch=PVE_START_SCRATCH)
    CHECK_REPORT = check_environment(PVE_INITIAL,PVE_STUDENT_DIR,PVE_POOL[0],[1],phase/'check')
    CHECKED_PVE_CONFIG = copy.deepcopy(PVE_CONFIG)
    CURRENT_AGENT = PVE_INITIAL
    show_result(CHECK_REPORT)
else:
    print('Environment check is OFF；训练前必须完成。')


### 4.1 Optional button test — no model learning
原课程的可选按键测试位置仍保留。这里查看当前动作表的六帧输入序列；实际有效性由第4节环境步进及第6/8节录像观察，不能仅凭名称认定招式已触发。


In [ ]:
RUN_ACTION_CHECK = False
if RUN_ACTION_CHECK:
    actor = load_actor(CURRENT_AGENT)
    try:
        for i, action in enumerate(actor.actions):
            print(i, action['name'], action['segments'])
    finally:
        actor.close()
else:
    print('Optional input-table review is OFF.')


## 5. PvE training — this cell updates the learner
打开原位置的`RUN_PVE_TRAINING`后，从第4节工作副本开始真实循环PPO训练。调用已有训练器，收集新对局，计算GAE，按连续序列更新网络。学习率、batch、rollout等取自第2节；反馈来自第3.3节。

这是权重初始化的新训练阶段，优化器重新开始，不是精确中断恢复。输出`initial.pt`、检查点、训练曲线数据及最后模型；不将未经完整评估的最后模型称作best。完成后同一个`CURRENT_AGENT`传给第6/8/9节，PvE与PvP不是两份互不相干的最终权重。

默认关闭训练；2048步只是检查接线的小预算，不能承诺增强。原版的训练完成也不等于策略更强。


In [ ]:
RUN_PVE_TRAINING = False
if RUN_PVE_TRAINING:
    if 'PVE_INITIAL' not in globals() or CHECKED_PVE_CONFIG != PVE_CONFIG:
        raise RuntimeError('Run Section 4 with the current PvE settings first.')
    LAST_TRAINING = new_run(ROOT,'pve_train')
    CURRENT_AGENT, TRAIN_REPORT = train_ppo(PVE_INITIAL,PVE_STUDENT_DIR,PVE_POOL,
                                           PVE_TRAIN_CONFIG,LAST_TRAINING/'training',ports=[1])
    print('PVE_TRAINING_COMPLETE；最后模型：', CURRENT_AGENT)
    print('真实更新次数：',TRAIN_REPORT['optimizer_updates'],'；原生帧：',TRAIN_REPORT['native_frames'])
else:
    print('PvE training is OFF. 当前模型：', CURRENT_AGENT)


### 5.1 Review training and the verified-trajectory integration method
保留原训练复盘位置，显示真正的训练曲线。另提供F最近一轮所用的历史连续轨迹整合入口：固定视觉/GRU，利用已验证整局轨迹监督训练动作读出层。

该历史复现需要Windows上保留的原始轨迹库，从历史配置指定的父模型开始，不是继续当前PPO模型；缺数据会明确报错。结果单独保存，不自动覆盖`CURRENT_AGENT`。不能把重复拟合旧轨迹称作已实现持续学习。


In [ ]:
RUN_TRAINING_REVIEW = False
RUN_REPRODUCE_TRAJECTORY_INTEGRATION = False
INTEGRATION_CONFIG = REPO_ROOT/'configs/ryu_targeted_delivery.json'
INTEGRATION_VARIANT = 'fresh_feedback_integration'
INTEGRATION_ITERATIONS = 1024
if RUN_TRAINING_REVIEW and TRAIN_REPORT:
    show_training_review(TRAIN_REPORT)
if RUN_REPRODUCE_TRAJECTORY_INTEGRATION:
    integration_run = new_run(ROOT,'historical_integration')
    INTEGRATION_MODEL = reproduce_integration(INTEGRATION_CONFIG,INTEGRATION_VARIANT,
                                               INTEGRATION_ITERATIONS,integration_run/'recipe')
    print('历史整合结果：',INTEGRATION_MODEL,'；未替换CURRENT_AGENT。')
if not RUN_TRAINING_REVIEW and not RUN_REPRODUCE_TRAJECTORY_INTEGRATION:
    print('Training review and historical integration are OFF.')


## 6. Independent PvE evaluation — no learning
沿用课程独立评估章节。默认使用当前工作模型，现场录制八星Ryu、八星Sagat两场；打开`EVALUATE_ALL_CPU=True`评估12角色×8星（默认不为96场全部录像）。

胜负在首次终局冻结，随后保留至少一秒真实动画。原生CPU在尾段仍由游戏控制；尾段不参与评分。这里不更新任何权重。


In [ ]:
RUN_EVALUATION = True
EVALUATION_BUNDLE = ''  # 空：当前工作模型；也可填已知可信的.pt或本地policy.zip
EVALUATE_ALL_CPU = False
EVALUATION_CONFIG = dict(record_video=True, post_match_seconds=POST_MATCH_SECONDS)
if RUN_EVALUATION:
    chosen = PVE_POOL if EVALUATE_ALL_CPU else [r for r in PVE_POOL
                                               if r['character'] in ('Ryu','Sagat') and r['stars']==8]
    if not chosen:
        raise ValueError('当前PvE配置不含演示条件；调整列表或启用完整CPU评估。')
    cases = [dict(title=f"PvE · {r['character']} {r['stars']} stars",opponent=r,player=1,offset=0) for r in chosen]
    LAST_EVALUATION = new_run(ROOT,'pve_evaluate')
    EVALUATION_REPORT = evaluate_cases(EVALUATION_BUNDLE or CURRENT_AGENT,cases,LAST_EVALUATION/'evaluation',
                                       record_video=EVALUATION_CONFIG['record_video'] and not EVALUATE_ALL_CPU,
                                       post_match_seconds=EVALUATION_CONFIG['post_match_seconds'])
    show_result(EVALUATION_REPORT)
else:
    print('Independent CPU evaluation is OFF.')


## 7. Package a trained model — local inference export
保留课程导出位置。导出当前模型的`policy.zip`、模型源码、观察与动作定义；使用真实`PPO.load`及已有隔离重载检查。与课程兼容的输入和六帧动作保持不变。

由于禁止执行文件哈希，本版不调用课程原来的外层submission打包及验证流程。本地推理导出不等于已完成官方Arena验收，也不伪造其验证字段。


In [ ]:
RUN_SUBMISSION_PACKAGING = False  # 本版对应本地推理导出
SUBMISSION_CONFIG = dict(team_id='group10', agent_path='')
if RUN_SUBMISSION_PACKAGING:
    export_run = new_run(ROOT,'export')
    LAST_SUBMISSION = export_model(SUBMISSION_CONFIG['agent_path'] or CURRENT_AGENT,export_run/'runtime')
    print('本地PPO推理模型：',LAST_SUBMISSION)
    display(FileLink(str(LAST_SUBMISSION)))
else:
    print('Local model export is OFF.')


### 7.1 Optional model picker — an alternative to Section 7
保留模型选择位置，改为显式路径，避免混淆历史best、last及当前权重。默认只显示路径；若要导出，回到第7节填写`agent_path`。


In [ ]:
SHOW_SUBMISSION_PICKER = False
if SHOW_SUBMISSION_PICKER:
    print('当前工作模型：',CURRENT_AGENT)
    print('本轮起点F：',DEFAULT_AGENT)
    if LAST_SUBMISSION:
        print('本次最近导出：',LAST_SUBMISSION)
    print('请在第7节SUBMISSION_CONFIG["agent_path"]选择明确路径。')
else:
    print('Optional model path review is OFF.')


### 7.2 Self-check the exported model — no learning
保留原自检位置，但只检查本地推理导出的结果，读取导出器已完成的独立80输入及96次连续推理检查。没有运行课程官方外层包检查，不能输出`ARENA_SELF_CHECK_PASSED`。


In [ ]:
RUN_SUBMISSION_SELF_CHECK = False
SUBMISSION_TO_CHECK = ''
if RUN_SUBMISSION_SELF_CHECK:
    selected = SUBMISSION_TO_CHECK or LAST_SUBMISSION
    if not selected:
        raise ValueError('先在第7节导出，或填写本地policy.zip路径。')
    selected = Path(selected)
    receipt = selected.parent/'reload_result.json'
    if not receipt.is_file():
        raise FileNotFoundError('此目录没有独立重载记录；请通过第7节重新导出。')
    result = json.loads(receipt.read_text(encoding='utf-8'))
    print({k:result[k] for k in ('observations','greedy_actions_equal','recurrent_sequence')})
    print('LOCAL_RELOAD_CHECK；不是官方Arena验收。')
else:
    print('Local export review is OFF.')


### 7.3 Optional backup — keep your experiment, not just one entry
保留备份位置，提供本次工作目录、实际权重和源码路径。大录像、训练缓存保留在`artifacts/`，不自动重复打包或提交Git。需要比赛接口的本地推理目录由第7节生成。


In [ ]:
RUN_EXPORT = False
if RUN_EXPORT:
    print('本次实验目录：',ROOT)
    print('当前模型：',CURRENT_AGENT)
    display(FileLink(str(CURRENT_AGENT)))
    if LAST_SUBMISSION:
        display(FileLink(str(LAST_SUBMISSION)))
    print('学生源文件：',PVE_STUDENT_DIR,PVP_STUDENT_DIR)
else:
    print('Backup path display is OFF.')


## 8. Match two saved agents — NO training
沿用原版“两份保存的agent对打”：A默认当前Counter-Ryu，B默认已有Ryu神经反制者。`swap_sides=True`分别运行A为P1和P2；每端口一局是本notebook的演示配置，不是对最终比赛赛制的断言。

两方权重都不更新。画面标明我方控制器身份，越过对手后左右位置会改变。录像保留终局后至少一秒；输入名称和位移不自动代表技能命中或有效战术。


In [ ]:
RUN_MATCH = True
MATCH_CONFIG = dict(
    agent_a='',
    agent_b='models/character_algorithm_breakthrough/final_counters/prospective_validation_counter_frozen_candidate_Ryu_59101_250000.pt',
    swap_sides=True, rounds_per_side=1, offset=12,
)
if RUN_MATCH:
    opponent_actor = load_actor(MATCH_CONFIG['agent_b'])
    opponent_character = opponent_actor.definition.get('character','Ryu')
    opponent_actor.close()
    rival = dict(id='classroom_frozen_opponent',kind='agent',character=opponent_character,path=MATCH_CONFIG['agent_b'])
    cases = [dict(title=f'PvP · Counter-Ryu P{port}',opponent=rival,player=port,offset=MATCH_CONFIG['offset'])
             for port in ([1,2] if MATCH_CONFIG['swap_sides'] else [1])
             for _ in range(MATCH_CONFIG['rounds_per_side'])]
    LAST_MATCH = new_run(ROOT,'match')
    MATCH_REPORT = evaluate_cases(MATCH_CONFIG['agent_a'] or CURRENT_AGENT,cases,LAST_MATCH/'matches',
                                  record_video=True,post_match_seconds=POST_MATCH_SECONDS)
    show_result(MATCH_REPORT)
else:
    print('Match is OFF. This section never updates either model.')


## 9. Optional PvP training — one learner, one fixed opponent
保留完整PvP训练路线。固定保存的对手不更新；只有当前Counter-Ryu学习。`learner_bundle=''`承接本notebook当前权重，所以若第5节已训练，PvP继续同一模型，不另起一份最终模型。

### 9.1 Select the learner and opponent
`learner_sides=[1,2]`在训练回合中采样两端口。可同时采样第2节CPU任务，减轻PvP阶段完全不接触PvE经验的问题；这不是已经证明“不遗忘”的算法。通过`include_pve`和明确的任务采样比例控制。

默认小预算只检查训练流程；新能力与保持效果必须用完整对局评价。


In [ ]:
PVP_AGENT_CONFIG = dict(team_name='group10',character='Ryu',seed=111037,use_position=True)
PVP_SETUP = dict(
    learner_start='package', learner_bundle='',
    opponent_bundle='models/character_algorithm_breakthrough/final_counters/prospective_validation_counter_frozen_candidate_Ryu_59101_250000.pt',
    learner_sides=[1,2], opponent_deterministic=True, neutral_jitter_max=0,
    include_pve=True,
)
PVP_TRAIN_CONFIG = dict(
    total_timesteps=2048,n_envs=4,n_steps=512,sequence_length=32,batch_size=256,n_epochs=4,
    learning_rate=1e-5,gamma=1.0,gae_lambda=0.95,clip_range=0.1,ent_coef=0.005,
    target_kl=0.04,device='cpu',checkpoint_every=2048,max_seconds=1800,
    seed=PVP_AGENT_CONFIG['seed'],coverage_offsets=[0,12],
    training_group_weights=dict(pve=0.25,p1=0.375,p2=0.375),
)
PVP_VALIDATION_CONFIG = dict(offsets=[0,12],record_video=False)
if PVP_AGENT_CONFIG['character'] != 'Ryu' or not PVP_AGENT_CONFIG['use_position']:
    raise ValueError('当前工作模型保持Ryu和坐标输入。')
if not PVP_SETUP['opponent_deterministic'] or PVP_SETUP['neutral_jitter_max'] != 0:
    raise ValueError('当前固定对手使用确定性动作；开局通过coverage_offsets设置。')
print('PvP learner:',PVP_AGENT_CONFIG)
print('PvP setup:',PVP_SETUP)
print('PPO settings:',PVP_TRAIN_CONFIG)


### 9.2 PvP student files — run all four cells
保留原版独立四格，写入`student/pvp/`。修改PvE文件不会悄悄改变这些文件。两套文件是训练配置入口，不表示最终保存两份策略；第9.4节仍续训同一个`CURRENT_AGENT`。

#### 9.2a Observation — independent PvP input definition
与当前F一致的84×96四帧输入。改完重跑9.3，不能绕过形状及兼容性检查。


In [ ]:
%%writefile $PVP_STUDENT_DIR/student_observation.py
"""Student-editable image input. Default: four 96 x 84 grayscale frames.

Only transform the supplied RGB image. Never operate the game or change rewards.
The coordinate branch still uses the six original self-first position values.
SB3 divides these uint8 images by 255 once, inside its policy preprocessor.
"""
import cv2
import numpy as np

INPUT_CONFIG = {'color': 'gray', 'width': 96, 'height': 84, 'stack_frames': 4, 'raw_shape': [224, 320, 3], 'crop_xyxy': [0, 0, 256, 224], 'verify_right_padding': True, 'black_threshold': 5, 'max_nonblack_fraction': 0.005}


def preprocess(rgb):
    """One RGB uint8 frame -> (channels, height, width) uint8 array."""
    a = np.asarray(rgb)
    c = INPUT_CONFIG
    if a.dtype != np.uint8 or tuple(a.shape) != tuple(c["raw_shape"]):
        raise ValueError(f"Unexpected raw image: {a.shape} {a.dtype}; expected {c['raw_shape']} uint8.")
    x0, y0, x1, y1 = c["crop_xyxy"]
    if c["verify_right_padding"] and x1 < a.shape[1]:
        excluded = a[:, x1:, :]
        fraction = float(np.mean(np.max(excluded, axis=2) > c["black_threshold"]))
        if fraction > c["max_nonblack_fraction"]:
            raise ValueError(
                f"CROP_GUARD_FAILED: right-side nonblack fraction={fraction:.6f}. "
                "The fixed viewport is not valid for this observed frame. "
                "Stop and inspect the raw image; do not train with a silently changing crop."
            )
    view = a[y0:y1, x0:x1, :]
    if c["color"] == "gray":
        gray = cv2.cvtColor(view, cv2.COLOR_RGB2GRAY)
        result = cv2.resize(gray, (c["width"], c["height"]), interpolation=cv2.INTER_AREA)[None, :, :]
    else:
        result = cv2.resize(view, (c["width"], c["height"]), interpolation=cv2.INTER_AREA).transpose(2, 0, 1)
    return np.ascontiguousarray(result, dtype=np.uint8)


#### 9.2b Actions — independent PvP action table
仍为64个六帧输入，编号与加载模型一致。固定对手使用自己的动作目录，不受本cell修改影响。


In [ ]:
%%writefile $PVP_STUDENT_DIR/student_actions.py
"""Group10: 15 course basics plus the 49 deployed six-frame inputs."""
CUSTOM_ACTIONS = [{'name': 'NO_OP_X_tap',
  'segments': [{'keys': [], 'frames': 2}, {'keys': ['X'], 'frames': 2}, {'keys': [], 'frames': 2}]},
 {'name': 'NO_OP_Y_tap',
  'segments': [{'keys': [], 'frames': 2}, {'keys': ['Y'], 'frames': 2}, {'keys': [], 'frames': 2}]},
 {'name': 'NO_OP_Z_tap',
  'segments': [{'keys': [], 'frames': 2}, {'keys': ['Z'], 'frames': 2}, {'keys': [], 'frames': 2}]},
 {'name': 'NO_OP_A_tap',
  'segments': [{'keys': [], 'frames': 2}, {'keys': ['A'], 'frames': 2}, {'keys': [], 'frames': 2}]},
 {'name': 'NO_OP_B_tap',
  'segments': [{'keys': [], 'frames': 2}, {'keys': ['B'], 'frames': 2}, {'keys': [], 'frames': 2}]},
 {'name': 'NO_OP_C_tap',
  'segments': [{'keys': [], 'frames': 2}, {'keys': ['C'], 'frames': 2}, {'keys': [], 'frames': 2}]},
 {'name': 'LEFT_X_tap',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'X'], 'frames': 2},
               {'keys': ['LEFT'], 'frames': 2}]},
 {'name': 'LEFT_Y_tap',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'Y'], 'frames': 2},
               {'keys': ['LEFT'], 'frames': 2}]},
 {'name': 'LEFT_Z_tap',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'Z'], 'frames': 2},
               {'keys': ['LEFT'], 'frames': 2}]},
 {'name': 'LEFT_A_tap',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'A'], 'frames': 2},
               {'keys': ['LEFT'], 'frames': 2}]},
 {'name': 'LEFT_B_tap',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'B'], 'frames': 2},
               {'keys': ['LEFT'], 'frames': 2}]},
 {'name': 'LEFT_C_tap',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'C'], 'frames': 2},
               {'keys': ['LEFT'], 'frames': 2}]},
 {'name': 'RIGHT_X_tap',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'X'], 'frames': 2},
               {'keys': ['RIGHT'], 'frames': 2}]},
 {'name': 'RIGHT_Y_tap',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'Y'], 'frames': 2},
               {'keys': ['RIGHT'], 'frames': 2}]},
 {'name': 'RIGHT_Z_tap',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'Z'], 'frames': 2},
               {'keys': ['RIGHT'], 'frames': 2}]},
 {'name': 'RIGHT_A_tap',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'A'], 'frames': 2},
               {'keys': ['RIGHT'], 'frames': 2}]},
 {'name': 'RIGHT_B_tap',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'B'], 'frames': 2},
               {'keys': ['RIGHT'], 'frames': 2}]},
 {'name': 'RIGHT_C_tap',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'C'], 'frames': 2},
               {'keys': ['RIGHT'], 'frames': 2}]},
 {'name': 'DOWN_X_tap',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'X'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2}]},
 {'name': 'DOWN_Y_tap',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'Y'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2}]},
 {'name': 'DOWN_Z_tap',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'Z'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2}]},
 {'name': 'DOWN_A_tap',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'A'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2}]},
 {'name': 'DOWN_B_tap',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'B'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2}]},
 {'name': 'DOWN_C_tap',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'C'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2}]},
 {'name': 'UP_X_tap',
  'segments': [{'keys': ['UP'], 'frames': 2},
               {'keys': ['UP', 'X'], 'frames': 2},
               {'keys': ['UP'], 'frames': 2}]},
 {'name': 'UP_Y_tap',
  'segments': [{'keys': ['UP'], 'frames': 2},
               {'keys': ['UP', 'Y'], 'frames': 2},
               {'keys': ['UP'], 'frames': 2}]},
 {'name': 'UP_Z_tap',
  'segments': [{'keys': ['UP'], 'frames': 2},
               {'keys': ['UP', 'Z'], 'frames': 2},
               {'keys': ['UP'], 'frames': 2}]},
 {'name': 'UP_A_tap',
  'segments': [{'keys': ['UP'], 'frames': 2},
               {'keys': ['UP', 'A'], 'frames': 2},
               {'keys': ['UP'], 'frames': 2}]},
 {'name': 'UP_B_tap',
  'segments': [{'keys': ['UP'], 'frames': 2},
               {'keys': ['UP', 'B'], 'frames': 2},
               {'keys': ['UP'], 'frames': 2}]},
 {'name': 'UP_C_tap',
  'segments': [{'keys': ['UP'], 'frames': 2},
               {'keys': ['UP', 'C'], 'frames': 2},
               {'keys': ['UP'], 'frames': 2}]},
 {'name': 'UP_LEFT_X_tap',
  'segments': [{'keys': ['UP', 'LEFT'], 'frames': 2},
               {'keys': ['UP', 'LEFT', 'X'], 'frames': 2},
               {'keys': ['UP', 'LEFT'], 'frames': 2}]},
 {'name': 'UP_LEFT_Y_tap',
  'segments': [{'keys': ['UP', 'LEFT'], 'frames': 2},
               {'keys': ['UP', 'LEFT', 'Y'], 'frames': 2},
               {'keys': ['UP', 'LEFT'], 'frames': 2}]},
 {'name': 'UP_LEFT_Z_tap',
  'segments': [{'keys': ['UP', 'LEFT'], 'frames': 2},
               {'keys': ['UP', 'LEFT', 'Z'], 'frames': 2},
               {'keys': ['UP', 'LEFT'], 'frames': 2}]},
 {'name': 'UP_RIGHT_X_tap',
  'segments': [{'keys': ['UP', 'RIGHT'], 'frames': 2},
               {'keys': ['UP', 'RIGHT', 'X'], 'frames': 2},
               {'keys': ['UP', 'RIGHT'], 'frames': 2}]},
 {'name': 'UP_RIGHT_Y_tap',
  'segments': [{'keys': ['UP', 'RIGHT'], 'frames': 2},
               {'keys': ['UP', 'RIGHT', 'Y'], 'frames': 2},
               {'keys': ['UP', 'RIGHT'], 'frames': 2}]},
 {'name': 'UP_RIGHT_Z_tap',
  'segments': [{'keys': ['UP', 'RIGHT'], 'frames': 2},
               {'keys': ['UP', 'RIGHT', 'Z'], 'frames': 2},
               {'keys': ['UP', 'RIGHT'], 'frames': 2}]},
 {'name': 'NO_OP_PPP',
  'segments': [{'keys': ['X', 'Y', 'Z'], 'frames': 2}, {'keys': [], 'frames': 4}]},
 {'name': 'QCF_RIGHT_Y',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'Y'], 'frames': 2}]},
 {'name': 'QCF_RIGHT_B',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'B'], 'frames': 2}]},
 {'name': 'QCF_RIGHT_C',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'RIGHT'], 'frames': 2},
               {'keys': ['RIGHT', 'C'], 'frames': 2}]},
 {'name': 'DP_RIGHT_X',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'RIGHT', 'X'], 'frames': 2}]},
 {'name': 'DP_RIGHT_Z',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'RIGHT', 'Z'], 'frames': 2}]},
 {'name': 'DP_RIGHT_B',
  'segments': [{'keys': ['RIGHT'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'RIGHT', 'B'], 'frames': 2}]},
 {'name': 'QCF_LEFT_Y',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'Y'], 'frames': 2}]},
 {'name': 'QCF_LEFT_B',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'B'], 'frames': 2}]},
 {'name': 'QCF_LEFT_C',
  'segments': [{'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'LEFT'], 'frames': 2},
               {'keys': ['LEFT', 'C'], 'frames': 2}]},
 {'name': 'DP_LEFT_X',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'LEFT', 'X'], 'frames': 2}]},
 {'name': 'DP_LEFT_Z',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'LEFT', 'Z'], 'frames': 2}]},
 {'name': 'DP_LEFT_B',
  'segments': [{'keys': ['LEFT'], 'frames': 2},
               {'keys': ['DOWN'], 'frames': 2},
               {'keys': ['DOWN', 'LEFT', 'B'], 'frames': 2}]}]


#### 9.2c Reward — measured from the learner's perspective
保留原课程奖励接口。无论学习者控制P1还是P2，`damage_dealt`、`damage_taken`和胜负都按学习者定义。这份代码会实际用于第9.4节的新训练阶段。


In [ ]:
%%writefile $PVP_STUDENT_DIR/student_reward.py
"""Simple classroom reward: small net-damage feedback + clear match outcome.

Compatible with SF2_Flexible_Classroom_PPO (ordinary PPO, six-frame steps).
The arena supplies damage_dealt/damage_taken with negative KO HP treated as 0.
A natural timeout uses the arena's win/loss/draw result, just like a KO.
A safety cutoff is not a loss. No distance, survival, action or time bonus.

With <=176 HP per fighter and no in-round healing, the sum of damage rewards
is bounded by [-0.5, 0.5]. A normal completed win therefore has positive raw
episode return, and a loss negative raw episode return. This is NOT a claim
about discounted returns, win rate, or empirically verified learning success.
"""
import math

REWARD_VERSION = "net_damage_outcome_v1"
FULL_HP = 176.0
DAMAGE_WEIGHT = 0.5
OUTCOME_BONUS = {
    "ongoing": 0.0,
    "win": 1.0,
    "loss": -1.0,
    "draw": 0.0,
    "cutoff": 0.0,
}


def compute_reward(t: dict) -> float:
    dealt = float(t["damage_dealt"])
    taken = float(t["damage_taken"])
    result = t["result"]

    # Do not silently replace invalid readings with a plausible reward.
    if not (math.isfinite(dealt) and math.isfinite(taken)):
        raise ValueError("Damage must be finite.")
    if not (0.0 <= dealt <= FULL_HP and 0.0 <= taken <= FULL_HP):
        raise ValueError("Damage is outside the expected 0..176 range.")
    if result not in OUTCOME_BONUS:
        raise ValueError(f"Unknown match result: {result!r}")

    # Count this step's damage even on the final step, then add outcome once.
    damage_reward = DAMAGE_WEIGHT * (dealt - taken) / FULL_HP
    return float(damage_reward + OUTCOME_BONUS[result])


#### 9.2d Model — independent PvP learner configuration
接入同一循环PPO结构与实际输入历史。配置用于当前学习者，不改变固定对手。若第5节已经产生新权重，默认在该权重上继续。


In [ ]:
%%writefile $PVP_STUDENT_DIR/student_model.py
"""Group10 recurrent PPO model settings used by prepare_student and the trainer."""
import copy

MODEL_CONFIG = {
    'kind': 'sequence_ppo',
    'memory_hidden': 128,
    'trainable_visual': True,
    'physical_actions': True,
    'signed_relative_position': True,
    'facing_canonical': True,
    'action_readout': 'relu',
    'action_temperature': 1.0,
    'input_history': {'mode': 'real', 'cap_native_frames': 120, 'normalization': 120},
}

def configure_definition(definition):
    result = copy.deepcopy(definition)
    result.update(copy.deepcopy(MODEL_CONFIG))
    return result


### 9.3 PvP environment check — required before PvP training
保留原独立PvP检查，默认对所选两端口显示输入及步进历史。学习者与对手各自持有记忆，不能把一方状态用于另一方。检查仅验证接口，不宣称完整对战能力。


In [ ]:
RUN_PVP_CHECK = True
if RUN_PVP_CHECK:
    if PVP_SETUP['learner_start'] not in ('package','scratch'):
        raise ValueError('learner_start须为package或scratch。')
    phase = new_run(ROOT,'pvp_check')
    PVP_INITIAL = prepare_student(PVP_STUDENT_DIR,PVP_SETUP['learner_bundle'] or CURRENT_AGENT,
                                  phase/'student',scratch=PVP_SETUP['learner_start']=='scratch')
    opponent_actor = load_actor(PVP_SETUP['opponent_bundle'])
    PVP_OPPONENT = dict(id='training_frozen_opponent',kind='agent',
                        character=opponent_actor.definition.get('character','Ryu'),path=PVP_SETUP['opponent_bundle'])
    opponent_actor.close()
    PVP_CHECK_REPORT = check_environment(PVP_INITIAL,PVP_STUDENT_DIR,PVP_OPPONENT,
                                         PVP_SETUP['learner_sides'],phase/'check')
    CURRENT_AGENT = PVP_INITIAL
    CHECKED_PVP_SETUP = copy.deepcopy(PVP_SETUP)
    show_result(PVP_CHECK_REPORT)
else:
    print('PvP check is OFF. Run 9.1 and all four 9.2 cells first.')


### 9.4 PvP training — only the learner changes
打开本节开关后，用当前四个PvP学生模块进行循环PPO训练。对手始终固定，采样端口与CPU混合比例取自9.1。学习者新权重保存到本次输出目录；结束后更新`CURRENT_AGENT`，第6节也可重新评估同一模型的PvE能力。

输出最后模型，不伪造一个未经评估的best。即使训练奖励上升，也必须通过9.5及CPU复测判断完整对局是否改善。


In [ ]:
RUN_PVP_TRAINING = False
if RUN_PVP_TRAINING:
    if 'PVP_INITIAL' not in globals() or CHECKED_PVP_SETUP != PVP_SETUP:
        raise RuntimeError('Run Section 9.3 with the current settings first.')
    pool = [PVP_OPPONENT] + (PVE_POOL if PVP_SETUP['include_pve'] else [])
    LAST_PVP_TRAINING = new_run(ROOT,'pvp_train')
    CURRENT_AGENT,PVP_TRAIN_REPORT = train_ppo(PVP_INITIAL,PVP_STUDENT_DIR,pool,
                                              PVP_TRAIN_CONFIG,LAST_PVP_TRAINING/'training',
                                              ports=PVP_SETUP['learner_sides'])
    print('PVP_TRAINING_COMPLETE；同一当前权重：',CURRENT_AGENT)
    print('更新次数：',PVP_TRAIN_REPORT['optimizer_updates'],'；原生帧：',PVP_TRAIN_REPORT['native_frames'])
else:
    print('PvP training is OFF. 当前模型：',CURRENT_AGENT)


### 9.5 Evaluate and review PvP training — no learning
保留训练前后比较。当前阶段起点与最后模型面对相同对手、端口和offset，分别报告完整对局。不能以最初模型作为best掩盖最后模型退化；也不能把当前开发条件重跑称作新的泛化证据。

训练后可重跑第6节检查CPU，同一份权重兼顾两类任务。


In [ ]:
RUN_PVP_EVALUATION = False
PVP_EVALUATION_BUNDLE = ''
PVP_REFERENCE_BUNDLE = ''
if RUN_PVP_EVALUATION:
    if 'PVP_OPPONENT' not in globals():
        raise RuntimeError('先运行9.3准备独立对手。')
    cases = [dict(opponent=PVP_OPPONENT,player=p,offset=o)
             for p in PVP_SETUP['learner_sides'] for o in PVP_VALIDATION_CONFIG['offsets']]
    phase = new_run(ROOT,'pvp_evaluate')
    PVP_EVALUATION_REPORT = evaluate_cases(PVP_EVALUATION_BUNDLE or CURRENT_AGENT,cases,phase/'current',
                                          record_video=PVP_VALIDATION_CONFIG['record_video'],post_match_seconds=POST_MATCH_SECONDS)
    show_result(PVP_EVALUATION_REPORT)
    reference = PVP_REFERENCE_BUNDLE or (str(PVP_INITIAL) if PVP_TRAIN_REPORT else '')
    if reference:
        PVP_BEFORE_REPORT = evaluate_cases(reference,cases,phase/'before',record_video=False)
        print('同条件阶段起点：')
        show_result(PVP_BEFORE_REPORT)
else:
    print('PvP evaluation is OFF.')


In [ ]:
RUN_PVP_REVIEW = False
if RUN_PVP_REVIEW and PVP_TRAIN_REPORT:
    show_training_review(PVP_TRAIN_REPORT)
else:
    print('PvP training review is OFF or no training occurred in this session.')


## 10. Troubleshooting and instructor checks

| 问题 | 处理 |
|---|---|
| ROOT、配置或模型未定义 | 按原章节顺序运行；PvE 1–4，PvP 1及9.1–9.3 |
| 某一步打印OFF | 与原版一样由对应RUN开关控制；默认不训练 |
| 修改学生文件后训练报错 | 重新执行写文件cell及环境检查，避免训练读取旧副本 |
| 架构/输入/动作不匹配 | 保留当前F配置，或显式从头初始化；不能无损加载任意新结构 |
| 原生模拟器或ROM缺失 | 使用已配置的sye6308环境与现有本地ROM安装入口，不执行原课程扫描流程 |
| 想训练后看效果 | 5或9.4完成后重跑6/8/9.5；它们使用同一个CURRENT_AGENT |
| 想复现F最后一次监督整合 | 使用5.1历史复现入口，需要Windows原始连续轨迹库 |
| 输出变差 | 记录完整对局与行为；训练完成不等于能力积累，不自动覆盖F |
| 想改CNN/GRU实现 | 3.4/9.2d给出共享实现位置；不要在notebook复制第二套训练器 |

### Instructor release checklist
本版来自课程原53格结构，保持游戏裁判、六帧接口、P1/P2身份规则。部署的四帧观察与输入历史均为合法信息。原课程公共安装及外层提交流程因用户禁止文件哈希而未执行；第7节只是本地推理交付，官方Arena仍需按课程接口另行解决。

### Current method and remaining work
F已有已知条件下的较强完整对局表现，但仍常重复攻击，尚未稳定学会广泛的主动走位与跨打法适应。五天研究的局部技能可作为后续经验，不是已完成持续改进。下一步研究能力A→B→C的积累机制；本次重写仅将已实现的方法正确接回课程notebook，不提前宣称新算法成功。
